# Mango Detection and Ripeness Classification

This notebook trains two models:
1. A YOLOv8 detector to locate mangoes in images.
2. A TensorFlow classifier to predict ripeness for cropped mango images.

The workflow is organized into setup, dataset preparation, training, evaluation, export, and app deployment steps.


In [1]:
# Install the core libraries used in this notebook.
# Run this cell once in a fresh Colab environment.
!pip install -q ultralytics roboflow tensorflow streamlit opencv-python kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 128.9 MB/s eta 0:00:00


## 1. Import Libraries and Define Project Settings

Keep project-wide paths and settings in one place so the notebook is easier to maintain.


In [2]:
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import tensorflow as tf
from IPython.display import Image, display
from roboflow import Roboflow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from ultralytics import YOLO

# Roboflow configuration.
# For better security, store the key in an environment variable when possible.
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "Lkq2ojJa2S9mNjGNuviJ")
WORKSPACE_NAME = "pigrecipe"
PROJECT_NAME = "mango-7l7ug-napwv-razzf"
PROJECT_VERSION = 1

# Local project paths used throughout the notebook.
DATASET_DIR = Path("mango-1")
YOLO_DATA_CONFIG = DATASET_DIR / "data.yaml"
YOLO_BASE_WEIGHTS = "yolov8n.pt"
YOLO_RUNS_DIR = Path("/content/runs/detect")
YOLO_TRAIN_DIR = YOLO_RUNS_DIR / "train"
YOLO_BEST_WEIGHTS = YOLO_TRAIN_DIR / "weights" / "best.pt"
PREDICT_DIR = YOLO_RUNS_DIR / "predict"
CROP_OUTPUT_DIR = Path("/content/mango_crops")
RIPENESS_DATA_DIR = Path("/content/ripeness_data")
CLASSIFIER_MODEL_PATH = Path("/content/mango_classifier.h5")

# Shared inference settings. These thresholds make the pipeline more conservative.
CLASS_NAMES = ["raw", "ripe", "overripe"]
DETECTION_CONF_THRESH = 0.5
CLASSIFICATION_CONF_THRESH = 0.6
CROP_MARGIN = 10
CLASSIFIER_IMAGE_SIZE = (224, 224)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Build Robust Inference Utilities

These helper functions make the pipeline reusable and safer. They add negative-detection handling, confidence filtering, margin-based cropping, and a single end-to-end image processing function.


In [3]:
def detect_mangoes(model, img: np.ndarray, conf_thresh: float = DETECTION_CONF_THRESH) -> List[Tuple[int, int, int, int, float]]:
    """Return high-confidence mango boxes from a detector prediction."""
    results = model.predict(img, conf=conf_thresh, verbose=False)
    boxes = []

    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            continue

        xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, conf in zip(xyxy, confidences):
            if float(conf) < conf_thresh:
                continue

            x1, y1, x2, y2 = map(int, box)
            boxes.append((x1, y1, x2, y2, float(conf)))

    return boxes


def crop_mango(img: np.ndarray, box: Tuple[int, int, int, int, float], margin: int = CROP_MARGIN) -> np.ndarray:
    """Crop a mango with a small safety margin to reduce tight-box errors."""
    x1, y1, x2, y2, _ = box
    height, width = img.shape[:2]

    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(width, x2 + margin)
    y2 = min(height, y2 + margin)

    return img[y1:y2, x1:x2]


def preprocess_crop(crop: np.ndarray) -> np.ndarray:
    """Resize and normalize a crop before classification."""
    resized = cv2.resize(crop, CLASSIFIER_IMAGE_SIZE).astype("float32") / 255.0
    return np.expand_dims(resized, axis=0)


def classify_mango(model, crop: np.ndarray, conf_thresh: float = CLASSIFICATION_CONF_THRESH) -> Tuple[str, float]:
    """Classify a crop and return an uncertain label when confidence is too low."""
    prediction = model.predict(preprocess_crop(crop), verbose=False)[0]
    confidence = float(np.max(prediction))
    label_idx = int(np.argmax(prediction))

    if confidence < conf_thresh:
        return "uncertain", confidence

    return CLASS_NAMES[label_idx], confidence


def process_image(
    img: np.ndarray,
    det_model,
    clf_model,
    det_conf_thresh: float = DETECTION_CONF_THRESH,
    clf_conf_thresh: float = CLASSIFICATION_CONF_THRESH,
    margin: int = CROP_MARGIN,
):
    """Detect mangoes, classify confident crops, and annotate the image."""
    annotated = img.copy()
    boxes = detect_mangoes(det_model, annotated, conf_thresh=det_conf_thresh)

    if len(boxes) == 0:
        return None, {"message": "No mango detected", "counts": {name: 0 for name in CLASS_NAMES}, "avg_detection_conf": 0.0}

    counts = {name: 0 for name in CLASS_NAMES}
    per_mango_results = []
    combined_scores = []

    for box in boxes:
        crop = crop_mango(annotated, box, margin=margin)
        if crop.size == 0:
            continue

        label, cls_conf = classify_mango(clf_model, crop, conf_thresh=clf_conf_thresh)
        x1, y1, x2, y2, det_conf = box
        combined_conf = float(det_conf * cls_conf)

        if label != "uncertain":
            counts[label] += 1

        per_mango_results.append({
            "box": box,
            "label": label,
            "detection_conf": float(det_conf),
            "classification_conf": float(cls_conf),
            "combined_conf": combined_conf,
        })
        combined_scores.append(combined_conf)

        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated,
            f"{label} | det:{det_conf:.2f} cls:{cls_conf:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    summary = {
        "message": "ok",
        "counts": counts,
        "avg_detection_conf": float(np.mean([box[4] for box in boxes])) if boxes else 0.0,
        "avg_combined_conf": float(np.mean(combined_scores)) if combined_scores else 0.0,
        "results": per_mango_results,
    }
    return annotated, summary


## 3. Download the Detection Dataset from Roboflow


In [4]:
# Connect to Roboflow and download the YOLOv8-ready dataset.
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(PROJECT_VERSION).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to mango-1 in yolov8:: 100%|██████████| 2005/2005 [00:00<00:00, 6910.96it/s]

Dataset downloaded to: /content/mango-1


In [5]:
# Inspect the extracted dataset structure to confirm the expected folders exist.
!ls
!ls mango-1


mango-1  sample_data
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


## 4. Train the YOLOv8 Mango Detector


In [6]:
# Initialize a lightweight YOLOv8 model and fine-tune it on the mango dataset.
detector = YOLO(YOLO_BASE_WEIGHTS)

detector.train(
    data=str(YOLO_DATA_CONFIG),
    epochs=50,
    imgsz=640,
    batch=16,
)


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=mango-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d2557034ef0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## 5. Run Detection Inference on Test Images


In [7]:
# Generate predictions on the test split and save visualized outputs.
detector.predict(
    source=str(DATASET_DIR / "test" / "images"),
    save=True,
    conf=DETECTION_CONF_THRESH,
)



image 1/50 /content/mango-1/test/images/320324240_1312732282914241_4069157750509378719_n_png_jpg.rf.00a06061d4846498817d83ccca114e2c.jpg: 640x640 2 Fresh Mangos, 5.8ms
image 2/50 /content/mango-1/test/images/633-2-_jpg.rf.43053bfef419273d7ee08834587da6e5.jpg: 640x640 1 Fresh Mango, 5.9ms
image 3/50 /content/mango-1/test/images/Apple-Mango-174-_jpg.rf.9df5fb162043cb71f1906e8d17eed785.jpg: 640x640 1 Fresh Mango, 5.8ms
image 4/50 /content/mango-1/test/images/Fresh-144_jpg.rf.a929dfa9fbd216294caa0fd637fdd138.jpg: 640x640 1 Fresh Mango, 5.8ms
image 5/50 /content/mango-1/test/images/Fresh-161_jpg.rf.534f308266b3108784767a5374a48c59.jpg: 640x640 3 Fresh Mangos, 5.8ms
image 6/50 /content/mango-1/test/images/Fresh-257_jpg.rf.df38ea117c308c7c49fa00b1cbc16257.jpg: 640x640 2 Fresh Mangos, 5.8ms
image 7/50 /content/mango-1/test/images/Fresh-259_jpg.rf.186765df05a7209738a69ce4233a659b.jpg: 640x640 1 Fresh Mango, 5.9ms
image 8/50 /content/mango-1/test/images/Fresh-26_jpg.rf.735928db7e41288c6a8be0a6f

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Fresh Mango'}
 obb: None
 orig_img: array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0]

## 6. Save Training Outputs to Google Drive

This step is optional, but useful if you want to keep training results after the Colab runtime resets.


In [8]:
from google.colab import drive

# Mount Google Drive and copy training runs for persistence.
drive.mount('/content/drive')
!cp -r /content/runs/detect/train* /content/drive/MyDrive/mango_project/


ValueError: mount failed

## 7. Load the Best Detector and Review Results


In [ ]:
# Reload the best trained weights for a clean evaluation step.
detector = YOLO(str(YOLO_BEST_WEIGHTS))

detector.predict(
    source=str(DATASET_DIR / "test" / "images"),
    save=True,
    conf=DETECTION_CONF_THRESH,
)

!ls /content/runs/detect/
!ls /content/runs/detect/predict/


In [ ]:
# Display the training summary plot created by Ultralytics.
Image(filename=str(YOLO_TRAIN_DIR / "results.png"))


In [ ]:
# Preview a few predicted images to visually inspect detection quality.
prediction_images = os.listdir(PREDICT_DIR)

for image_name in prediction_images[:5]:
    display(Image(filename=str(PREDICT_DIR / image_name)))


## 8. Crop Detected Mangoes for Ripeness Classification

The detector output is reused to build a crop set that can later be classified by ripeness. Using the shared helper functions keeps crop generation consistent with inference.


In [ ]:
# Create a directory to store detected mango crops.
CROP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

crop_count = 0
train_image_paths = sorted((DATASET_DIR / "train" / "images").glob("*"))

for image_path in train_image_paths:
    image_array = cv2.imread(str(image_path))
    if image_array is None:
        continue

    boxes = detect_mangoes(detector, image_array, conf_thresh=DETECTION_CONF_THRESH)

    for box in boxes:
        crop = crop_mango(image_array, box, margin=CROP_MARGIN)

        # Skip invalid or empty crops to avoid write errors.
        if crop.size == 0:
            continue

        crop_path = CROP_OUTPUT_DIR / f"mango_{crop_count}.jpg"
        cv2.imwrite(str(crop_path), crop)
        crop_count += 1

print(f"Saved {crop_count} mango crops to {CROP_OUTPUT_DIR}")


## 9. Download the Ripeness Classification Dataset


In [ ]:
# Download and extract the ripeness classification dataset from Kaggle.
!kaggle datasets download -d srabon00/mango-ripening-stage-classification
!unzip -q mango-ripening-stage-classification.zip -d /content/ripeness_data


## 10. Prepare TensorFlow Data Loaders


In [ ]:
# Use a validation split so we can monitor generalization during training.
datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,
)

train_data = datagen.flow_from_directory(
    str(RIPENESS_DATA_DIR),
    target_size=CLASSIFIER_IMAGE_SIZE,
    batch_size=32,
    subset="training",
)

val_data = datagen.flow_from_directory(
    str(RIPENESS_DATA_DIR),
    target_size=CLASSIFIER_IMAGE_SIZE,
    batch_size=32,
    subset="validation",
)


## 11. Build and Train the Ripeness Classifier


In [ ]:
# Start from EfficientNetB0 for transfer learning.
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)

# Freeze the backbone for the initial training pass.
base_model.trainable = False

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(128, activation="relu")(x)
classifier_output = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

classifier = tf.keras.Model(inputs=base_model.input, outputs=classifier_output)
classifier.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

classifier.summary()


In [ ]:
# Train the classifier on the ripeness dataset.
classifier.fit(train_data, validation_data=val_data, epochs=10)


## 12. Test the Full Pipeline on a Sample Image

This uses the same end-to-end logic that the app will use, including no-mango handling and confidence-aware ripeness classification.


In [ ]:
# Run the complete detection + classification pipeline on one test image.
sample_test_image_path = sorted((DATASET_DIR / "test" / "images").glob("*"))[0]
sample_bgr = cv2.imread(str(sample_test_image_path))
processed_image, summary = process_image(sample_bgr, detector, classifier)

if processed_image is None:
    print(summary["message"])
else:
    display(Image(filename=str(sample_test_image_path)))
    print("Counts:", summary["counts"])
    print(f"Average detection confidence: {summary['avg_detection_conf']:.2f}")
    print(f"Average combined confidence: {summary['avg_combined_conf']:.2f}")


In [ ]:
# Load one cropped mango image and predict its ripeness label with confidence filtering.
sample_image = image.load_img(CROP_OUTPUT_DIR / "mango_0.jpg", target_size=CLASSIFIER_IMAGE_SIZE)
sample_array = image.img_to_array(sample_image)
label, confidence = classify_mango(classifier, sample_array)

print(f"Predicted class: {label} | confidence: {confidence:.2f}")


## 13. Export Trained Models


In [ ]:
# Copy the best detector weights and save the classification model.
!cp /content/runs/detect/train/weights/best.pt /content/
classifier.save(CLASSIFIER_MODEL_PATH)


## 14. Negative Detection Improvement Plan

The current logic can now return `No mango detected`, but detector robustness still depends on training data. To reduce false positives in real-world images, retrain YOLO with a stronger negative set:

- Add background-only images with no mango annotations.
- Include hard negatives such as other fruits, leaves, branches, baskets, and random indoor objects.
- Keep these images unlabeled so YOLO learns that no box should be predicted.
- Re-evaluate precision on a validation folder that mixes mango and non-mango scenes.

This is the most important model-level step for making `No mango` reliable.


## 15. Build a Streamlit App

The following cell writes a demo app that uses the same robust inference rules as the notebook.


In [ ]:
%%writefile app.py
import cv2
import numpy as np
import streamlit as st
from tensorflow.keras.models import load_model
from ultralytics import YOLO

DETECTION_MODEL_PATH = "best.pt"
CLASSIFICATION_MODEL_PATH = "mango_classifier.h5"
CLASS_NAMES = ["raw", "ripe", "overripe"]
DETECTION_CONF_THRESH = 0.5
CLASSIFICATION_CONF_THRESH = 0.6
CROP_MARGIN = 10
IMAGE_SIZE = (224, 224)


@st.cache_resource
def load_models():
    """Load the detection and classification models once per session."""
    detection_model = YOLO(DETECTION_MODEL_PATH)
    classification_model = load_model(CLASSIFICATION_MODEL_PATH)
    return detection_model, classification_model


def detect_mangoes(model, img, conf_thresh=DETECTION_CONF_THRESH):
    """Return high-confidence mango boxes from a detector prediction."""
    results = model.predict(img, conf=conf_thresh, verbose=False)
    boxes = []

    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            continue

        xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, conf in zip(xyxy, confidences):
            if float(conf) < conf_thresh:
                continue

            x1, y1, x2, y2 = map(int, box)
            boxes.append((x1, y1, x2, y2, float(conf)))

    return boxes


def crop_mango(img, box, margin=CROP_MARGIN):
    """Crop a mango with a small safety margin to reduce tight-box errors."""
    x1, y1, x2, y2, _ = box
    height, width = img.shape[:2]

    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(width, x2 + margin)
    y2 = min(height, y2 + margin)

    return img[y1:y2, x1:x2]


def preprocess_crop(crop):
    """Resize and normalize a crop before classification."""
    resized = cv2.resize(crop, IMAGE_SIZE).astype("float32") / 255.0
    return np.expand_dims(resized, axis=0)


def classify_mango(model, crop, conf_thresh=CLASSIFICATION_CONF_THRESH):
    """Classify a crop and return an uncertain label when confidence is too low."""
    prediction = model.predict(preprocess_crop(crop), verbose=False)[0]
    confidence = float(np.max(prediction))
    label_idx = int(np.argmax(prediction))

    if confidence < conf_thresh:
        return "uncertain", confidence

    return CLASS_NAMES[label_idx], confidence


def process_image(img, det_model, clf_model):
    """Detect mangoes, classify confident crops, and annotate the image."""
    annotated = img.copy()
    boxes = detect_mangoes(det_model, annotated)

    if len(boxes) == 0:
        return None, {
            "message": "No mango detected",
            "counts": {name: 0 for name in CLASS_NAMES},
            "avg_detection_conf": 0.0,
            "avg_combined_conf": 0.0,
        }

    counts = {name: 0 for name in CLASS_NAMES}
    combined_scores = []

    for box in boxes:
        crop = crop_mango(annotated, box)
        if crop.size == 0:
            continue

        label, cls_conf = classify_mango(clf_model, crop)
        x1, y1, x2, y2, det_conf = box
        combined_conf = float(det_conf * cls_conf)
        combined_scores.append(combined_conf)

        if label != "uncertain":
            counts[label] += 1

        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            annotated,
            f"{label} | det:{det_conf:.2f} cls:{cls_conf:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    summary = {
        "message": "ok",
        "counts": counts,
        "avg_detection_conf": float(np.mean([box[4] for box in boxes])) if boxes else 0.0,
        "avg_combined_conf": float(np.mean(combined_scores)) if combined_scores else 0.0,
    }
    return annotated, summary


def main():
    st.set_page_config(page_title="Mango Detection and Ripeness App", layout="centered")
    st.title("Mango Detection and Ripeness App")
    st.write("Upload an image to detect mangoes and estimate ripeness for each detected fruit.")

    detection_model, classification_model = load_models()
    uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

    if uploaded_file is None:
        return

    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image_array = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    st.image(image_array, caption="Uploaded image", channels="BGR", use_container_width=True)

    processed_image, summary = process_image(image_array, detection_model, classification_model)

    if processed_image is None:
        st.error("No mango detected")
        return

    total_mangoes = sum(summary["counts"].values())
    st.image(processed_image, caption="Detection and ripeness result", channels="BGR", use_container_width=True)
    st.write(f"Total classified mangoes: {total_mangoes}")
    st.write(summary["counts"])
    st.write(f"Average detection confidence: {summary['avg_detection_conf']:.2f}")
    st.write(f"Average combined confidence: {summary['avg_combined_conf']:.2f}")


if __name__ == "__main__":
    main()
